## SQL Questions

In [21]:
# Connecting to the project_1 database that we have created
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",                     
    password="Assasin#96",         
    database="project_1"             
)
cursor = conn.cursor()
print("Connected to project_1")

Connected to project_1


In [22]:
# Helper: run a SQL query and return the result as a pandas table
def run_query(sql):
    cursor.execute(sql)
    return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

#### Q1: List all movies in the Action genre released after 2015, showing title and release date.

In [36]:
q1 = """
SELECT m.title, m.release_date
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g        ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
  AND YEAR(m.release_date) > 2015
ORDER BY m.release_date
"""
a1=run_query(q1)
print(f"Output: \n{a1}\n")

print(f"=> {len(a1)} Action movies were released after 2015, ranging from {min(a1['release_date'])} to {max(a1['release_date'])}. ")

Output: 
                            title release_date
0                    The 5th Wave   2016-01-14
1                 Kung Fu Panda 3   2016-01-23
2                     Zoolander 2   2016-02-06
3                        Deadpool   2016-02-09
4                        Triple 9   2016-02-19
..                            ...          ...
263                            F1   2025-06-25
264                      Superman   2025-07-09
265  The Fantastic 4: First Steps   2025-07-23
266            Predator: Badlands   2025-11-05
267              Mortal Kombat II   2026-05-06

[268 rows x 2 columns]

=> 268 Action movies were released after 2015, ranging from 2016-01-14 to 2026-05-06. 


#### Q2: Find the top 10 highest-grossing movies along with the genre(s) each belongs to.

In [24]:
q2 = """
SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name SEPARATOR ', ') AS genres
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g        ON mg.genre_id = g.genre_id
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10
"""
run_query(q2)

,title,revenue,genres
0,Avatar,2923706026,"Adventure, Science Fiction, Action"
1,Avengers: Endgame,2799439100,"Science Fiction, Action, Adventure"
2,Avatar: The Way of Water,2334484620,"Action, Science Fiction, Adventure"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Science Fiction, Adventure, Action"
5,Avengers: Infinity War,2052415039,"Adventure, Science Fiction, Action"
6,Spider-Man: No Way Home,1921206586,"Adventure, Action, Science Fiction"
7,Zootopia 2,1868208796,"Family, Mystery, Animation, Adventure, Comedy"
8,Inside Out 2,1698863816,"Family, Comedy, Adventure, Animation"
9,Jurassic World,1671537444,"Science Fiction, Thriller, Adventure"


#### Q3: Calculate the average budget and average revenue for each genre, sorted by average revenue descending.

In [ ]:
q3 = """
SELECT g.genre_name,
       ROUND(AVG(m.budget))  AS avg_budget, 
       ROUND(AVG(m.revenue)) AS avg_revenue
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g        ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY avg_revenue DESC
"""
run_query(q3)

,genre_name,avg_budget,avg_revenue
0,Adventure,109359964,390214464
1,Animation,83360605,359743487
2,Family,84584257,343096752
3,Science Fiction,95259923,323560110
4,Action,93294567,300054700
5,Fantasy,86177842,294297494
6,Comedy,52569069,211351278
7,Music,41069388,197452578
8,Romance,34611126,162789647
9,Thriller,49387696,160515200


#### Q4: List the top 10 actors who have appeared in the most movies in the dataset.(cast)

In [40]:
q4 = """
SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
FROM cast
GROUP BY person_id, actor_name
ORDER BY movie_count DESC
LIMIT 10
"""
run_query(q4)

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Mark Wahlberg,32
6,Willem Dafoe,32
7,Scarlett Johansson,32
8,Tom Cruise,31
9,Morgan Freeman,31


#### Q5: Find all directors who have directed more than 3 movies, along with their movie count.

In [41]:
q5 = """
SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC
"""
run_query(q5)

,director,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Robert Eggers,4
213,Ryan Coogler,4
214,Andy Muschietti,4
215,Wes Ball,4


#### Q6: List the top 10 most frequently used keywords across all movies.

In [42]:
q6 = """
SELECT keyword_name, COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
GROUP BY keyword_id, keyword_name
ORDER BY movie_count DESC
LIMIT 10
"""
run_query(q6)

,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


#### Q7: Find movies where the budget was above the overall average budget but the revenue was below the overall average revenue.

In [43]:
q7 = """
SELECT title, budget, revenue
FROM movies
WHERE budget  > (SELECT AVG(budget)  FROM movies)
  AND revenue > 0
  AND revenue < (SELECT AVG(revenue) FROM movies)
ORDER BY budget DESC
"""
run_query(q7)

,title,budget,revenue
0,The Marvels,274800000,206136825
1,Red One,250000000,185700759
2,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
3,Mulan,200000000,69965374
4,Wonder Woman 1984,200000000,169601036
...,...,...,...
296,Nightmare Alley,60000000,39629195
297,Damsel,60000000,5000
298,A Haunting in Venice,60000000,121400000
299,Gran Turismo,60000000,121700000


#### Q8: List all actors who have appeared in a movie directed by a specific director 

In [52]:
q8 = """
SELECT cr.person_name as Director, m.title as Movie_Title, group_concat(c.actor_name SEPARATOR ', ') as Actors
FROM cast c
join movies m on m.movie_id =c.movie_id
JOIN crew cr ON c.movie_id = cr.movie_id
WHERE cr.job = 'Director'
GROUP BY Director, Movie_Title
ORDER BY Director, Movie_Title
"""
run_query(q8)

,Director,Movie_Title,Actors
0,Aaron Blaise,Brother Bear,"Joan Copeland, Joaquin Phoenix, Jeremy Suarez,..."
1,Aaron Horvath,The Super Mario Bros. Movie,"Charles Martinet, Seth Rogen, Keegan-Michael K..."
2,Aaron Horvath,The Super Mario Galaxy Movie,"Anya Taylor-Joy, Glen Powell, Luis Guzmán, Jac..."
3,Aaron Nee,The Lost City,"Daniel Radcliffe, Sandra Bullock, Da'Vine Joy ..."
4,Aaron Schneider,Greyhound,"Josh Wiggins, Stephen Graham, Chet Hanks, Karl..."
...,...,...,...
2733,Zack Snyder,Watchmen,"Malin Akerman, Patrick Wilson, Carla Gugino, J..."
2734,Zack Snyder,Zack Snyder's Justice League,"Willem Dafoe, Amy Adams, Ciarán Hinds, Ezra Mi..."
2735,Zak Hilditch,1922,"Thomas Jane, Molly Parker, Dylan Schmid, Kaitl..."
2736,Zhang Yimou,Hero,"Tony Leung Chiu-wai, Zhang Yakun, Qin Yan, Mag..."


#### Q9: Find the genre with the highest average rating, considering only genres with at least 20 movies.

In [46]:
q9 = """
SELECT g.genre_name, COUNT(*) AS total_movies, ROUND(AVG(m.vote_average), 2) AS avg_rating
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g        ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
HAVING COUNT(*) >= 20
ORDER BY avg_rating DESC
limit 1
"""
run_query(q9)

,genre_name,total_movies,avg_rating
0,War,85,7.36


#### Q10: List the top 10 movies with the highest number of keywords tagged to them. 

In [47]:
q10 = """
SELECT m.title, COUNT(DISTINCT mk.keyword_id) AS keyword_count
FROM movies m
JOIN movie_keywords mk ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10
"""
run_query(q10)

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,13 Hours: The Secret Soldiers of Benghazi,49
